<link rel="stylesheet" href="/site-assets/css/gemma.css">
<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Google+Symbols:opsz,wght,FILL,GRAD@20..48,100..700,0..1,-50..200" />

##### Copyright 2024 Google LLC。

In [ ]:
#@title Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# 使用 JAX 和 Flax 進行 RecurrentGemma 推論

<table class="tfo-notebook-buttons" align="left"> <td>    <a target="_blank" href="https://ai.google.dev/gemma/docs/recurrentgemma/recurrentgemma_jax_inference"><img src="https://ai.google.dev/static/site-assets/images/docs/notebook-site-button.png" height="32" width="32" />View on ai.google.dev</a>
</td> <td>    <a target="_blank" href="https://colab.research.google.com/github/google-gemma/cookbook/blob/main/docs/recurrentgemma/recurrentgemma_jax_inference.ipynb"><img src="https://www.tensorflow.org/images/colab_logo_32px.png" />Run in Google Colab</a>
</td> <td>    <a target="_blank" href="https://kaggle.com/kernels/welcome?src=https://github.com/google-gemma/cookbook/blob/main/docs/recurrentgemma/recurrentgemma_jax_inference.ipynb"><img src="https://www.kaggle.com/static/images/logos/kaggle-logo-transparent-300.png" height="32" width="70"/>Run in Kaggle</a>
</td> <td>    <a target="_blank" href="https://console.cloud.google.com/vertex-ai/colab/import/https%3A%2F%2Fraw.githubusercontent.com%2Fgoogle-gemma%2Fcookbook%2Fmain%2Fdocs%2Frecurrentgemma%2Frecurrentgemma_jax_inference.ipynb"><img src="https://ai.google.dev/images/cloud-icon.svg" width="40" />Open in Vertex AI</a>
</td> <td>    <a target="_blank" href="https://github.com/google-gemma/cookbook/blob/main/docs/recurrentgemma/recurrentgemma_jax_inference.ipynb"><img src="https://www.tensorflow.org/images/GitHub-Mark-32px.png" />View source on GitHub</a>
</td>
</table>

本教學示範如何使用 [RecurrentGemma](https://ai.google.dev/gemma/docs/recurrentgemma) 2B 指令模型執行基本取樣/inference，使用 [Google DeepMind 的 `recurrentgemma` library](@@P0002@P000@@library](@@P0002@P0000@@ (a 高效能數值計算library)、[Flax](https://flax.readthedocs.io)（基於JAX的神經網路@@P0011@ @）、[Orbax](https://orbax.readthedocs.io/)（基於JAX的library用於培訓實用程序，如checkpointing），和[SentencePiece](https://github.com/google/sentencepiece) (a tokenizer/detokenizer library)。雖然 Flax 在此 notebook 中沒有直接使用，但 Flax 用於創建 Gemma 和 RecurrentGemma（Griffin 模型）。
此 notebook 可以在具有 T4 GPU 的 Google Colab 上執行（轉至 **編輯** > **notebook 設定** > 在 **硬體加速器** 下選擇 **T4 GPU**）。

## 設定

以下部分介紹了準備 notebook 以使用 RecurrentGemma 模型的步驟，包括模型存取、獲取 API 金鑰以及設定 notebook runtime

### 為Gemma 設定Kaggle 存取權限

要完成本教學，您首先需要遵循類似[Gemma 設定](https://ai.google.dev/gemma/docs/setup) 的設定說明，但有些例外：
* 在 [kaggle.com](https://www.kaggle.com/models/google/recurrentgemma) 上造訪 RecurrentGemma（而非 Gemma）。
* 選擇具有足夠資源的 Colab runtime 來執行 RecurrentGemma 模型。
* 產生並設定 Kaggle 使用者名稱和 API 金鑰。

完成 RecurrentGemma 設定後，請前往下一部分，您將為 Colab 環境設定環境變數。
### 設定環境變數

設定`KAGGLE_USERNAME` 和`KAGGLE_KEY` 的環境變數。當 prompted 並選擇「授予存取權限？」時訊息，同意提供secret存取。

In [ ]:
import os
from google.colab import userdata # `userdata` is a Colab API.

os.environ["KAGGLE_USERNAME"] = userdata.get('KAGGLE_USERNAME')
os.environ["KAGGLE_KEY"] = userdata.get('KAGGLE_KEY')

### 安裝 `recurrentgemma` library

此notebook 重點關注使用免費的Colab GPU。若要啟用硬體加速，請按一下 **編輯** > **筆記型電腦設定** > 選擇 **T4 GPU** > **儲存**。
接下來，您需要從 [`github.com/google-deepmind/recurrentgemma`](https://github.com/google-deepmind/recurrentgemma) 安裝 Google DeepMind `recurrentgemma` library。如果您收到有關「pip 的依賴解析器」的錯誤，通常可以忽略它。
**附註：**透過安裝`recurrentgemma`，您也將安裝[`flax`](https://flax.readthedocs.io)、核心[`jax`](https://jax.readthedocs.io](@@P006@@)、核心[`jax`](https://jax.readthedocs.io](`optax`](@@P0008@0)（基於處理的梯度@P0103@P@P00008@P000008 [`orbax`](https://orbax.readthedocs.io/) 和 [`sentencepiece`](https://github.com/google/sentencepiece)。

In [ ]:
!pip install git+https://github.com/google-deepmind/recurrentgemma.git

  Cloning https://github.com/google-deepmind/recurrentgemma.git to /tmp/pip-req-build-zz9xp6s4
  Running command git clone --filter=blob:none --quiet https://github.com/google-deepmind/recurrentgemma.git /tmp/pip-req-build-zz9xp6s4
  Resolved https://github.com/google-deepmind/recurrentgemma.git to commit e4939f9b7edf8baa1d512fb86bfc2e206044d66b
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.6/44.6 kB 1.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.7/40.7 kB 3.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 20.8 MB/s eta 0:00:00
  Created wheel for recurrentgemma: filename=recurrentgemma-0.1.0-py3-none-any.whl size=73547 sha256=e3d3e85d59877ec33d2e4dff1a1666eaed1342c68199255cdd806d74472d4524
  Stored in directory: /tmp/pip-ephem-wheel-cache-42qdygtw/wheels/31/37/18/c57f1df6091b661385ab728b959bdfbf2078d

## 載入並準備 RecurrentGemma 模型

1. 使用 [`kagglehub.model_download`](https://github.com/Kaggle/kagglehub/blob/bddefc718182282882b72f814d407d89e5d178c4/src/kagglehub/models.py#L12) 載入 RecurrentGemma 模型，此模型採用三個參數：

- `handle`：來自Kaggle的模型句柄
- `path`：（可選字串）本地路徑
- `force_download`：（可選布林值）強制重新下載模型

**注意：** 請注意，`recurrentgemma-2b-it` 型號的大小約為 3.85Gb。

In [ ]:
RECURRENTGEMMA_VARIANT = '2b-it' # @param ['2b', '2b-it'] {type:"string"}

In [ ]:
import kagglehub

RECURRENTGEMMA_PATH = kagglehub.model_download(f'google/recurrentgemma/flax/{RECURRENTGEMMA_VARIANT}')

100%|██████████| 3.85G/3.85G [00:52<00:00, 78.2MB/s]
Extracting model files...


In [ ]:
print('RECURRENTGEMMA_PATH:', RECURRENTGEMMA_PATH)

RECURRENTGEMMA_PATH: /root/.cache/kagglehub/models/google/recurrentgemma/flax/2b-it/1


**注意：** 上面輸出的路徑是模型權重和tokenizer本地保存的位置，稍後您將需要它們。

2. 檢查模型權重和tokenizer的位置，然後設定路徑變數。 tokenizer 目錄將位於您下載模型的主目錄中，而模型權重將位於子目錄中。例如：

- `tokenizer.model` 檔案將位於 `/LOCAL/PATH/TO/recurrentgemma/flax/2b-it/1`)。
- 模型checkpoint 將位於`/LOCAL/PATH/TO/recurrentgemma/flax/2b-it/1/2b-it`)。

In [ ]:
CKPT_PATH = os.path.join(RECURRENTGEMMA_PATH, RECURRENTGEMMA_VARIANT)
TOKENIZER_PATH = os.path.join(RECURRENTGEMMA_PATH, 'tokenizer.model')
print('CKPT_PATH:', CKPT_PATH)
print('TOKENIZER_PATH:', TOKENIZER_PATH)

CKPT_PATH: /root/.cache/kagglehub/models/google/recurrentgemma/flax/2b-it/1/2b-it
TOKENIZER_PATH: /root/.cache/kagglehub/models/google/recurrentgemma/flax/2b-it/1/tokenizer.model


## 執行採樣/inference

1. 使用 [`recurrentgemma.jax.load_parameters`](https://github.com/google-deepmind/recurrentgemma/blob/e4939f9b7edf8baa1d512fb86bfc2e206044d66b/recurrentgemma/jax/utils.py#L31) 方法載入循環Gemma 模型checkpoint。設定為`"single_device"` 的`sharding` 參數會載入單一裝置上的所有模型參數。

In [ ]:
import recurrentgemma
from recurrentgemma import jax as recurrentgemma

params = recurrentgemma.load_parameters(checkpoint_path=CKPT_PATH, sharding="single_device")

2. 載入使用 [`sentencepiece.SentencePieceProcessor`](https://github.com/google/sentencepiece/blob/4d6a1f41069c4636c51a5590f7578a0dbed83450/python/src/sentencepiece/__init__.py#L423) 建構的循環Gemma 模型 tokenizer：

In [ ]:
import sentencepiece as spm

vocab = spm.SentencePieceProcessor()
vocab.Load(TOKENIZER_PATH)

True

3. 若要從 RecurrentGemma 型號 checkpoint 自動載入正確的設定，請使用 [`recurrentgemma.GriffinConfig.from_flax_params_or_variables`](https://github.com/google-deepmind/recurrentgemma/blob/e4939f9b7edf8baa1d512fb86bfc2e206044d66b/recurrentgemma/common.py#L128)。然後，使用 [`recurrentgemma.jax.Griffin`](https://github.com/google-deepmind/recurrentgemma/blob/e4939f9b7edf8baa1d512fb86bfc2e206044d66b/recurrentgemma/jax/griffin.py#L29) 實例化 [Griffin](https://arxiv.org/abs/2402.19427) 模型。

In [ ]:
model_config = recurrentgemma.GriffinConfig.from_flax_params_or_variables(
    flax_params_or_variables=params)

model = recurrentgemma.Griffin(model_config)

3. 在循環Gemma模型checkpoint/權重和tokenizer之上創建一個帶有[`recurrentgemma.jax.Sampler`](https://github.com/google-deepmind/recurrentgemma/blob/e4939f9b7edf8baa1d512fb86bfc2e206044d66b/recurrentgemma/jax/sampler.py#L74)的`sampler`：

In [ ]:
sampler = recurrentgemma.Sampler(
    model=model,
    vocab=vocab,
    params=params,
)

4. 在`prompt`中寫入prompt並執行inference。您可以調整`total_generation_steps`（產生回應時執行的步驟數 - 本範例使用`50`保留主機記憶體）。

**注意：**如果內存不足，請按一下**執行時** > **斷開連接並刪除runtime**，然後單擊**執行時** > **全部執行**。

In [ ]:
prompt = [
    "\n# 5+9=?",
]

reply = sampler(input_strings=prompt,
                total_generation_steps=50,
                )

for input_string, out_string in zip(prompt, reply.text):
    print(f"Prompt:\n{input_string}\nOutput:\n{out_string}")

/usr/local/lib/python3.10/dist-packages/jax/_src/interpreters/mlir.py:920: UserWarning: Some donated buffers were not usable: ShapedArray(int32[1,8]).
See an explanation at https://jax.readthedocs.io/en/latest/faq.html#buffer-donation.
  warnings.warn("Some donated buffers were not usable:"


Prompt:

# 5+9=?
Output:


# Answer: 14

# Explanation: 5 + 9 = 14.


## 了解更多

- 您可以了解有關 Google DeepMind [`recurrentgemma` library on GitHub](https://github.com/google-deepmind/recurrentgemma) 的更多信息，其中包含您在本教學中使用的方法和模組的文檔字符串，例如 [`recurrentgemma.jax.load_parameters`](@@P0050@P0050和[`recurrentgemma.jax.Sampler`](https://github.com/google-deepmind/recurrentgemma/blob/e4939f9b7edf8baa1d512fb86bfc2e206044d66b/recurrentgemma/jax/sampler.py#L74)。
- 以下庫有自己的文件網站：[core JAX](https://jax.readthedocs.io)、[Flax](https://flax.readthedocs.io) 和 [Orbax](https://orbax.readthedocs.io/)。
- 有關 `sentencepiece` tokenizer/detokenizer 文檔，請查看 [Google 的 `sentencepiece` GitHub 儲存庫](https://github.com/google/sentencepiece)。
- 對於 `kagglehub` 文檔，請查看 [Kaggle 的 `kagglehub` GitHub 存儲庫](https://github.com/Kaggle/kagglehub) 上的 `README.md`。
- 了解如何[將 Gemma 模型與 Google Cloud Vertex AI 一起使用](https://cloud.google.com/vertex-ai/docs/generative-ai/open-models/use-gemma)。
- 查看[循環Gemma：過去Transformers
高效開放語言模型](https://arxiv.org/pdf/2404.07839) Google DeepMind 的論文。- 閱讀 [Griffin：將門控線性遞歸與
Local Attention for Efficient Language Models](https://arxiv.org/pdf/2402.19427) GoogleDeepMind 的論文，以了解有關 RecurrentGemma 使用的模型架構的更多資訊。